# Step 3a: Simple baselines

Reference rankers that need no training. Each one scores every candidate in every impression, and candidates are ranked within their own impression. They establish what exposure counts, the candidate's own clicks, and related-content clicks achieve on their own, before any learned model.

| `model_id` | Score | What it establishes |
|---|---|---|
| `random` | constant score, so order comes only from the tie-break key | The floor any ranking must beat |
| `popularity` | `prior_exposures` | Ranking by how often the article was shown before. A reference method, not a reconstruction of MSN's ranker. |
| `category_ctr` | `category_prior_ctr` = prior CTR of other articles in the same category, shrunk toward the global CTR (50 pseudo-exposures) | Related-content behaviour alone. Content features must beat this to justify themselves. |
| `candidate_smoothed_ctr` | (prior_clicks + β·g<sub>t</sub>) / (prior_exposures + β) | The candidate's own clicks with generic smoothing toward the global CTR g<sub>t</sub> |
| `content_smoothed_ctr` | (prior_clicks + α·related_ctr) / (prior_exposures + α) | The same smoothing, but toward a content-specific prior (subcategory → category → global) |

**Information cutoff.** Every input column is computed from events strictly before the impression's timestamp (feature schema contract v0.1). Counters keep updating through val and test; nothing here is fitted.

**Zero-support handling.** An article with no prior exposures gets popularity 0, candidate smoothed CTR = g<sub>t</sub>, and content smoothed CTR = related_ctr. A category with no earlier events gets g<sub>t</sub>. Ties (common for popularity) are broken by a seeded hash of `row_id` that ignores labels and is the same for every model.

**Tuning.** β and α are chosen on **validation only**, over the same grid {5, 10, 20, 50, 100}. Selection rule (also used by the supplementary LightGBM steps 3c and 3d): highest validation Cold Recall@10 among settings within 0.005 validation nDCG@10 of the best, ties broken by nDCG@10. The tolerance is provisional until the team agrees it. The test split is scored once, at the end, with the frozen choices.

**Controlled comparison (handover A3).** `candidate_smoothed_ctr` and `content_smoothed_ctr` are compared at the **same strength** (α = β) for every grid value, and also at their own best values. Same candidates, splits, slates and cutoffs throughout.

**Outputs.** One prediction file per model, config and split with columns `row_id, model_id, config_id, predicted_score`, for Shi Ting's evaluator.

> Metrics in this notebook come from `shared/eval_interim.py`, a stand-in used only for tuning until the team evaluator exists. The numbers reported in the final report come from `step6_evaluation_diagnostics.ipynb`.

**At a glance**

| | |
|---|---|
| Reads | `features/v0.1/` on Google Drive `MyDrive/BT4222_G11/` |
| Writes | `predictions/real/`: five baselines on every split, plus the β and α validation grids |
| Time and hardware | About 1 minute on CPU |
| Run after | Step 2 |

## 1. Setup

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Bryng23/BT4222-Group-11.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/BT4222-Group-11"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    DATA_ROOT = "/content/drive/MyDrive/BT4222_G11"
else:
    REPO_DIR = os.environ.get("G11_REPO_DIR", os.getcwd())
    DATA_ROOT = os.environ.get("G11_DATA_ROOT", os.path.join(REPO_DIR, "data"))
sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
from shared import schema as S, derived as D, selection as SEL
from shared.predictions import write_predictions
from shared import eval_interim as E      # swap for the team evaluator once it exists

DATA_KIND = "REAL"   # recorded in every prediction file
FEATURE_ROOT = os.path.join(DATA_ROOT, "features", f"v{S.SCHEMA_VERSION}")
PRED_ROOT = os.path.join(DATA_ROOT, "predictions")

version = S.read_version(FEATURE_ROOT)
assert version.get("data_kind", "REAL") == DATA_KIND, f"feature tables are {version.get('data_kind')}, run is {DATA_KIND}"
print(f"DATA_KIND = {DATA_KIND} | features: {FEATURE_ROOT} | schema {version['schema_version']}")

DATA_KIND = REAL | features: C:\Users\Bryan Ng\g11_data\features\v0.1 | schema 0.1


## 2. Load only the columns the baselines use

Labels and cold flags come from the metadata table; scores come from the feature table. The two are aligned on `row_id`.

In [2]:
META_COLS = ["row_id", "impression_id", "label", "is_cold_10", "is_zero_exposure"]
FEAT_COLS = ["row_id", "prior_exposures", "prior_clicks", "global_prior_ctr", "category_prior_ctr", "related_ctr"]

data = {}
for split in S.SPLITS:
    m = pd.read_parquet(S.meta_path(FEATURE_ROOT, split), columns=META_COLS)
    f = pd.read_parquet(S.features_path(FEATURE_ROOT, split), columns=FEAT_COLS)
    assert np.array_equal(m["row_id"].to_numpy(), f["row_id"].to_numpy()), f"{split}: row_id misaligned"
    data[split] = (m, f)

pd.DataFrame({split: {"impressions": m["impression_id"].nunique(), "candidate rows": len(m),
                      "CTR": m["label"].mean(), "cold row share (<10)": m["is_cold_10"].mean(),
                      "clicked cold candidates": int((m["label"] * m["is_cold_10"]).sum())}
              for split, (m, f) in data.items()}).T.round(4)

,impressions,candidate rows,CTR,cold row share (<10),clicked cold candidates
train,95071.0,3389342.0,0.0418,0.0250,3549.0
val,31624.0,1231673.0,0.0389,0.0172,1005.0
test,30270.0,1222429.0,0.0383,0.0180,912.0


## 3. Scoring functions

In [3]:
GRID = S.ALPHA_GRID
TUNED = ("candidate_smoothed_ctr", "content_smoothed_ctr")

def score(model_id, f, strength=None):
    if model_id == "random":
        return np.zeros(len(f))
    if model_id == "popularity":
        return f["prior_exposures"].to_numpy(np.float64)
    if model_id == "category_ctr":
        return f["category_prior_ctr"].to_numpy(np.float64)
    if model_id == "candidate_smoothed_ctr":
        return D.candidate_smoothed_ctr(f, strength)
    if model_id == "content_smoothed_ctr":
        return D.content_smoothed_ctr(f, strength)
    raise ValueError(model_id)

def config_id(model_id, strength):
    if model_id == "candidate_smoothed_ctr":
        return f"beta={strength}"
    if model_id == "content_smoothed_ctr":
        return f"alpha={strength}"
    return "default"

SHOW = ["auc", "mrr", "ndcg@5", "ndcg@10", "cold_recall@10", "cold_hit@10", "warm_recall@10", "n_cold_click_impressions"]

## 4. Tune β and α on validation

Every grid value is scored on validation, and the prediction file is saved, so the team evaluator can reproduce the selection later.

In [4]:
val_meta, val_feat = data["val"]
val_per_imp, rows = {}, []
for model_id in TUNED:
    for s in GRID:
        sc = score(model_id, val_feat, s)
        write_predictions(val_meta["row_id"], sc, root=PRED_ROOT, data_kind=DATA_KIND, model_id=model_id,
                          config_id=config_id(model_id, s), split="val", schema_version=S.SCHEMA_VERSION)
        pi = E.per_impression_metrics(val_meta, sc)
        val_per_imp[(model_id, s)] = pi
        rows.append({"model_id": model_id, "strength": s, **E.summarise(pi)})
grid = pd.DataFrame(rows).set_index(["model_id", "strength"])
grid[SHOW].round(4)

auc     mrr  ndcg@5  ndcg@10  \
model_id               strength                                    
candidate_smoothed_ctr 5         0.6434  0.2975  0.3183   0.3773   
                       10        0.6441  0.2970  0.3178   0.3770   
                       20        0.6447  0.2962  0.3170   0.3762   
                       50        0.6447  0.2946  0.3151   0.3745   
                       100       0.6437  0.2925  0.3125   0.3724   
content_smoothed_ctr   5         0.6439  0.2978  0.3186   0.3776   
                       10        0.6445  0.2972  0.3179   0.3773   
                       20        0.6449  0.2966  0.3175   0.3769   
                       50        0.6447  0.2951  0.3155   0.3751   
                       100       0.6435  0.2931  0.3132   0.3730   

                                 cold_recall@10  cold_hit@10  warm_recall@10  \
model_id               strength                                                
candidate_smoothed_ctr 5                 0.2932       0.3017          0.6166   
                       10                0.2923       0.3006          0.6169   
                       20                0.2772       0.2825          0.6168   
                       50                0.2319       0.2356          0.6165   
                       100               0.1999       0.2026          0.6159   
content_smoothed_ctr   5                 0.3371       0.3454          0.6159   
                       10                0.3616       0.3699          0.6162   
                       20                0.3701       0.3795          0.6162   
                       50                0.3577       0.3657          0.6149   
                       100               0.3332       0.3401          0.6139   

                                 n_cold_click_impressions  
model_id               strength                            
candidate_smoothed_ctr 5                              938  
                       10                             938  
                       20                             938  
                       50                             938  
                       100                            938  
content_smoothed_ctr   5                              938  
                       10                             938  
                       20                             938  
                       50                             938  
                       100                            938

In [5]:
SELECT = SEL.PRIMARY
print("selection rule:", SEL.describe())
best = {}
for model_id in TUNED:
    g = grid.loc[model_id]
    top = SEL.select(g)
    best[model_id] = int(top)
    print(f"{model_id}: selected strength {top} "
          f"(val {SEL.PRIMARY} = {g.loc[top, SEL.PRIMARY]:.4f}, {SEL.GUARD} = {g.loc[top, SEL.GUARD]:.4f})")

selection rule: highest validation cold_recall@10 among settings whose validation ndcg@10 is within 0.005 of the best; ties broken on ndcg@10
candidate_smoothed_ctr: selected strength 5 (val cold_recall@10 = 0.2932, ndcg@10 = 0.3773)
content_smoothed_ctr: selected strength 20 (val cold_recall@10 = 0.3701, ndcg@10 = 0.3769)


**Takeaway: tuning.** The two smoothed-CTR scores rank almost identically overall: validation nDCG@10 stays between 0.372 and 0.378 for both across the whole grid. They differ only on cold candidates. With the global prior, Cold Recall@10 falls as β grows (0.29 → 0.20), because stronger smoothing pulls every low-exposure article toward the same global CTR. With the content prior it peaks at α = 20 (0.37), because shrinking toward the article's subcategory CTR keeps cold articles apart. The selection rule therefore picks β = 5 and α = 20.

## 5. Controlled comparison: content-specific prior vs global prior

For each strength s, compare `content_smoothed_ctr` (α = s) with `candidate_smoothed_ctr` (β = s) on the **same validation impressions**. The difference is content minus global. The 95% interval is a paired bootstrap over impressions (1,000 resamples), and the last column says how many impressions with a cold click support each Cold Recall@10 comparison.

A positive, stable difference with an interval above zero supports the conclusion: *better performance under these controlled conditions supports the usefulness of the content-specific prior for this task.* A difference whose interval crosses zero does not.

In [6]:
cmp_rows = []
pairs = [(f"alpha=beta={s}", s, s) for s in GRID]
pairs.append((f"best vs best (alpha={best['content_smoothed_ctr']}, beta={best['candidate_smoothed_ctr']})",
              best["content_smoothed_ctr"], best["candidate_smoothed_ctr"]))
for label, a, b in pairs:
    pc, pg = val_per_imp[("content_smoothed_ctr", a)], val_per_imp[("candidate_smoothed_ctr", b)]
    row = {"comparison": label}
    for metric in ("ndcg@10", "auc", "mrr"):
        row[f"Δ {metric}"] = pc[metric].mean() - pg[metric].mean()
    bs = E.paired_bootstrap(pg["cold_recall@10"], pc["cold_recall@10"])
    row.update({"Δ cold_recall@10": bs["mean_diff"], "95% CI low": bs["ci_low"], "95% CI high": bs["ci_high"],
                "cold-click impressions": bs["n_impressions"]})
    cmp_rows.append(row)
comparison = pd.DataFrame(cmp_rows).set_index("comparison")
comparison.round(4)

,Δ ndcg@10,Δ auc,Δ mrr,Δ cold_recall@10,95% CI low,95% CI high,cold-click impressions
comparison,,,,,,,
alpha=beta=5,0.0003,0.0004,0.0003,0.0439,0.0259,0.0618,938
alpha=beta=10,0.0003,0.0004,0.0001,0.0693,0.0490,0.0896,938
alpha=beta=20,0.0007,0.0002,0.0004,0.0929,0.0677,0.1187,938
alpha=beta=50,0.0007,0.0000,0.0005,0.1258,0.0975,0.1546,938
alpha=beta=100,0.0006,-0.0002,0.0006,0.1333,0.1045,0.1610,938
"best vs best (alpha=20, beta=5)",-0.0004,0.0014,-0.0009,0.0769,0.0531,0.1025,938


**Takeaway: controlled comparison.** At every equal strength, the content-specific prior raises validation Cold Recall@10 by 0.04 to 0.13, and every 95% interval is above zero (938 impressions with a cold click). The gain grows with strength, since the prior then carries more weight. nDCG@10, AUC and MRR move by less than 0.002. Under these controlled conditions the content-specific prior helps cold articles at no cost elsewhere, but it does not improve ranking overall.

## 6. Final scoring with frozen choices

The selections above are now fixed. Every baseline is scored on train, validation and test, and its prediction files are written. Train and validation are shown because the report must give training and testing performance; this is the first and only time the test split is scored.

In [7]:
FINAL = [("random", None), ("popularity", None), ("category_ctr", None),
         ("candidate_smoothed_ctr", best["candidate_smoothed_ctr"]),
         ("content_smoothed_ctr", best["content_smoothed_ctr"])]

final_rows, paths = [], []
for split in S.SPLITS:
    meta, feat = data[split]
    for model_id, s in FINAL:
        sc = score(model_id, feat, s)
        cfg = config_id(model_id, s)
        paths.append(write_predictions(meta["row_id"], sc, root=PRED_ROOT, data_kind=DATA_KIND, model_id=model_id,
                                       config_id=cfg, split=split, schema_version=S.SCHEMA_VERSION,
                                       extra={"selected_on": "val", "selection_metric": SELECT}))
        final_rows.append({"split": split, "model_id": model_id, "config_id": cfg, **E.evaluate(meta, sc)})
final = pd.DataFrame(final_rows).set_index(["split", "model_id"])
print(f"wrote {len(paths)} prediction files under {os.path.join(PRED_ROOT, DATA_KIND.lower())}")
final[["config_id"] + SHOW].round(4)

wrote 15 prediction files under C:\Users\Bryan Ng\g11_data\predictions\real


config_id     auc     mrr  ndcg@5  ndcg@10  \
split model_id                                                            
train random                   default  0.5000  0.2420  0.2512   0.3071   
      popularity               default  0.4917  0.2524  0.2651   0.3174   
      category_ctr             default  0.4822  0.2371  0.2437   0.2999   
      candidate_smoothed_ctr    beta=5  0.6784  0.3538  0.3823   0.4363   
      content_smoothed_ctr    alpha=20  0.6799  0.3535  0.3816   0.4362   
val   random                   default  0.5000  0.2182  0.2213   0.2779   
      popularity               default  0.4836  0.2192  0.2212   0.2755   
      category_ctr             default  0.4903  0.2085  0.2154   0.2708   
      candidate_smoothed_ctr    beta=5  0.6434  0.2975  0.3183   0.3773   
      content_smoothed_ctr    alpha=20  0.6449  0.2966  0.3175   0.3769   
test  random                   default  0.5000  0.2026  0.2034   0.2623   
      popularity               default  0.4608  0.1822  0.1792   0.2405   
      category_ctr             default  0.4977  0.1945  0.1929   0.2549   
      candidate_smoothed_ctr    beta=5  0.6468  0.2784  0.2937   0.3569   
      content_smoothed_ctr    alpha=20  0.6479  0.2775  0.2929   0.3563   

                              cold_recall@10  cold_hit@10  warm_recall@10  \
split model_id                                                              
train random                          0.2747       0.2867          0.5382   
      popularity                      0.0344       0.0348          0.5435   
      category_ctr                    0.2287       0.2358          0.5275   
      candidate_smoothed_ctr          0.2809       0.2938          0.6666   
      content_smoothed_ctr            0.3211       0.3335          0.6662   
val   random                          0.2725       0.2814          0.4996   
      popularity                      0.0309       0.0309          0.4897   
      category_ctr                    0.3324       0.3380          0.4984   
      candidate_smoothed_ctr          0.2932       0.3017          0.6166   
      content_smoothed_ctr            0.3701       0.3795          0.6162   
test  random                          0.2282       0.2378          0.4844   
      popularity                      0.0287       0.0287          0.4666   
      category_ctr                    0.2979       0.3059          0.4808   
      candidate_smoothed_ctr          0.2489       0.2593          0.5953   
      content_smoothed_ctr            0.2694       0.2784          0.5952   

                              n_cold_click_impressions  
split model_id                                          
train random                                      3223  
      popularity                                  3223  
      category_ctr                                3223  
      candidate_smoothed_ctr                      3223  
      content_smoothed_ctr                        3223  
val   random                                       938  
      popularity                                   938  
      category_ctr                                 938  
      candidate_smoothed_ctr                       938  
      content_smoothed_ctr                         938  
test  random                                       837  
      popularity                                   837  
      category_ctr                                 837  
      candidate_smoothed_ctr                       837  
      content_smoothed_ctr                         837

## 7. What the baselines show

* **Popularity is worse than random** (test AUC 0.46, Cold Recall@10 0.03). It ranks by prior exposures, so every zero-exposure article scores 0 and sinks to the bottom of its slate, which explains the near-zero cold recall. Among warm articles, more exposures do not mean more clicks either; a likely reason is that the most-shown articles are older by the time they appear. Exposure count is a poor proxy for interest in this log.
* **Category CTR alone carries little overall signal** (AUC about 0.49, nDCG@10 below random) but has the best test Cold Recall@10 of any baseline (0.30). Category behaviour is weak for ordering warm articles and useful for placing cold ones.
* **Random's Cold Recall@10 is 0.23 to 0.27, not near zero.** Many slates are short, so a clicked cold article often lands in the top 10 by chance. Read cold metrics against this floor: `candidate_smoothed_ctr` (0.25 on test) barely clears it.
* **The smoothed-CTR scores are the strongest baselines** (test AUC about 0.65, nDCG@10 about 0.36), and almost all of that comes from the candidate's own clicks. The content prior's cold advantage shrinks from validation (0.370 vs 0.293) to test (0.269 vs 0.249). The test difference is not bootstrapped here; step 6 should check whether it holds.
* **Nothing here is fitted**, so higher train numbers reflect easier train slates (random reaches nDCG@10 0.307 on train, 0.262 on test), not overfitting. Compare models within a split, not across splits.

*Based on the run with feature schema v0.1 and the interim evaluator. The final numbers and intervals come from step 6.*